In [0]:
%pip install xgboost scikit-learn mlflow

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 87.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 137.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 94.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 76.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 98.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 826.6/826.6 kB 34.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.4/206.4 MB 88.9 MB/s  0:00:02
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.13
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-97b457ac-d6e2-45fa-9064-151d8adc9ea9
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: blinker
    F

In [0]:
import mlflow
import mlflow.xgboost
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc
from xgboost import XGBClassifier

# 1. Apontar o Model Registry para o Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# 2. Carregar a tabela salva na etapa anterior
# (Confirme no menu Catalog se o nome exato é transacoes_features ou outro)
df = spark.read.table("workspace.default.transacoes_features").toPandas()

X = df.drop("FRAUD", axis=1).astype("float64")
y = df["FRAUD"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scale_pos_weight_value = float(y_train.value_counts()[0] / y_train.value_counts()[1])

# 3. Iniciar rastreamento no MLflow (fica salvo automaticamente em Experiments)
with mlflow.start_run(run_name="xgboost_cost_sensitive"):
    params = {
        "n_estimators": 100,
        "max_depth": 4,
        "scale_pos_weight": round(scale_pos_weight_value, 2),
        "eval_metric": "logloss",
        "random_state": 42
    }
    mlflow.log_params(params)

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    # Avaliação com AUPRC
    y_prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    auprc = auc(recall, precision)
    mlflow.log_metric("auprc", auprc)

    # Assinatura de entrada/saída (obrigatória no Unity Catalog)
    signature = infer_signature(X_train, model.predict(X_train))

    # Registro no Unity Catalog (sem hífens no nome)
    mlflow.xgboost.log_model(
        xgb_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(2),
        registered_model_name="workspace.default.fraude_cartao_xgboost"
    )

    print(f"Modelo registrado no Unity Catalog com AUPRC = {auprc:.4f}")

2026/11/23 19:25:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-1e4738dd-be82.cloud.databricks.com/ml/experiments/3986308451317604/models/m-a9afe949a81e4b3092228bad66972d70?o=7474660371209642
Successfully registered model 'workspace.default.fraude_cartao_xgboost'.


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.fraude_cartao_xgboost': https://dbc-1e4738dd-be82.cloud.databricks.com/explore/data/models/workspace/default/fraude_cartao_xgboost/version/1?o=7474660371209642


Modelo registrado no Unity Catalog com AUPRC = 1.0000


In [0]:
from mlflow import MlflowClient

client = MlflowClient()
model_name = "workspace.default.fraude_cartao_xgboost"

# Busca a última versão registrada e atribui o alias 'champion'
latest_version = max(
    int(v.version) for v in client.search_model_versions(f"name='{model_name}'")
)
client.set_registered_model_alias(model_name, "champion", str(latest_version))
print(f"Alias '@champion' atribuído à versão {latest_version} do modelo {model_name}.")

Alias '@champion' atribuído à versão 1 do modelo workspace.default.fraude_cartao_xgboost.
